In [14]:
import numpy as np

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, accuracy_score

from numba import njit

In [15]:
time = 1000  # Total seconds
dt = 0.0001
steps = int(time / dt)  # Total steps
tau_steps = int(0.7 / dt)  # How far we see back

test_size = 0.2

m = 1.0
k = 2.0
mu = 0.3

t = np.linspace(0, time, steps)
u = (
    t.astype(int) % 2
) * 2 - 1  # Basially just cut down just the int of time no dec then mod to get 0 or 1 every sec

In [27]:
@njit
def run_simulation(steps, dt, m, k, mu, u):
    x = np.zeros(steps)
    v = np.zeros(steps)
    x[0] = 1.0
    v[0] = 0.0

    for i in range(1, steps):
        x_dot = v[i - 1]
        v_dot = -mu / m * v[i - 1] - k / m * x[i - 1] + u[i - 1] / m

        x[i] = x[i - 1] + x_dot * dt
        v[i] = v[i - 1] + v_dot * dt

    return x, v


x, v = run_simulation(steps, dt, m, k, mu, u)

In [28]:
X = np.column_stack((x, v))
X_train, X_test, y_train, y_test = train_test_split(
    X[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
)

In [44]:
ridge_model = RidgeCV()
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

In [45]:
y_pred_discrete = np.where(y_pred_ridge > 0.0, 1, -1)

r_2 = r2_score(y_test, y_pred_ridge)
accuracy = accuracy_score(y_test, y_pred_discrete) * 100

print(f"{r_2:.4f}", f"{accuracy:.2f}%")

0.7582 98.96%


already has bias in the ridgecv

In [12]:
X = np.column_stack((x, v, np.ones_like(x)))
X_train, X_test, y_train, y_test = train_test_split(
    X[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
)

ridge_model = RidgeCV()
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

y_pred_discrete = np.where(y_pred_ridge > 0.0, 1, -1)

r_2 = r2_score(y_test, y_pred_ridge)
accuracy = accuracy_score(y_test, y_pred_discrete) * 100

print(f"{r_2:.4f}", f"{accuracy:.2f}%")

0.7582 98.96%


scaling worked

In [13]:
from sklearn.preprocessing import StandardScaler

# 1. Prepare your raw X (no bias column yet)
X = np.column_stack((x, v))

# 2. Normalize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. NOW add the bias column (ones) to the scaled data
X_final = np.column_stack((X_scaled, np.ones(X_scaled.shape[0])))

# 4. Now perform the split and training
X_train, X_test, y_train, y_test = train_test_split(
    X_final[tau_steps:],
    u[:-tau_steps],
    test_size=test_size,
    shuffle=False,  # Use shuffle=False for time series!
)

ridge_model = RidgeCV()
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

y_pred_discrete = np.where(y_pred_ridge > 0.0, 1, -1)

r_2 = r2_score(y_test, y_pred_ridge)
accuracy = accuracy_score(y_test, y_pred_discrete) * 100

print(f"{r_2:.4f}", f"{accuracy:.2f}%")

0.7962 99.73%
